In [5]:
import os
import pandas as pd
import numpy as np

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam


In [ ]:

# 시드 고정
seed = 0
np.random.seed(seed)
tf.random.set_seed(seed)

csv_path = "pokemon_gen1.csv"
img_dir = "PokemonData"

# CSV 불러오기
df = pd.read_csv(csv_path)

# --- 컬럼 체크 ---
if 'Name' not in df.columns:
    possible_name_cols = [c for c in df.columns if 'name' in c.lower()]
    if possible_name_cols:
        df.rename(columns={possible_name_cols[0]: 'Name'}, inplace=True)
    else:
        raise ValueError("CSV에 'Name' 컬럼이 존재하지 않습니다.")
if 'Type1' not in df.columns:
    raise ValueError("CSV에 'Type1' 컬럼이 존재하지 않습니다.")

# 이미지 경로 수집
image_paths = []
for name in df["Name"]:
    folder_path = os.path.join(img_dir, name)
    if os.path.exists(folder_path):
        for f in os.listdir(folder_path):
            if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.webp')):
                image_paths.append({"Name": name, "filepath": os.path.join(folder_path, f)})
image_df = pd.DataFrame(image_paths)
if image_df.empty:
    raise ValueError("이미지 파일이 존재하지 않거나 경로가 잘못되었습니다.")

# Type1 인코딩 (LabelEncoder)
df["Type1"] = df["Type1"].astype(str)
le = LabelEncoder()
df["Type1_enc"] = le.fit_transform(df["Type1"])
num_classes = len(le.classes_)

# merge
name_label_df = df[["Name", "Type1_enc"]]
merged = image_df.merge(name_label_df, on="Name", how="inner")
if merged.empty:
    raise ValueError("merge 결과가 비었습니다. Name 컬럼과 폴더/파일명이 일치하는지 확인하세요.")

# Type1_enc → 문자열로 변환
merged["Type1_str"] = merged["Type1_enc"].astype(str)

# 학습/검증 분리
train_df, val_df = train_test_split(merged, test_size=0.2, random_state=42, shuffle=True)


### 라벨 저장: 한 번만 실행 (학습 스크립트 안)
import json
with open("labels.json", "w", encoding="utf-8") as f:
    json.dump(label_cols, f, ensure_ascii=False, indent=2)
print("labels.json 저장 완료:", len(label_cols), "classes")


# 데이터 제너레이터
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
target_size = (256,256)
batch_size = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.25,
    shear_range=12,
    horizontal_flip=True,
    brightness_range=(0.75,1.25)
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="Type1_str",        # 문자열
    target_size=target_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True
)
validation_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="filepath",
    y_col="Type1_str",
    target_size=target_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)


classes = np.unique(train_generator.classes)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_generator.classes)
class_weight = dict(zip(classes, weights))


# 모델 구성 (MobileNetV2 백본)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(256,256,3))
base_model.trainable = False

from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.losses import CategoricalCrossentropy

x_in = base_model.output
x_in = GlobalAveragePooling2D()(x_in)
x_in = BatchNormalization()(x_in)
x_in = Dense(256, activation='swish')(x_in)   # ← Swish 활성화 적용
x_in = Dropout(0.35)(x_in)
predictions = Dense(num_classes, activation='softmax')(x_in)

model = Model(inputs=base_model.input, outputs=predictions)

# 라벨 스무딩 추가 (권장)
loss = CategoricalCrossentropy(label_smoothing=0.05)

model.compile(optimizer=Adam(1e-4), loss=loss, metrics=['accuracy'])

# 콜백
ckpt_path = "best_pokemon_type1.keras"
checkpoint = ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True, mode='min', verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=7, mode='min', restore_best_weights=True, verbose=1)

# 학습
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=1000,
    callbacks=[checkpoint, early_stopping],
    class_weight=class_weight
)

# 최종 저장
model.save('pokemon_type1_multiclass.keras')

# ======================================
# 새로운 이미지 예측 함수
# ======================================
from tensorflow.keras.preprocessing import image
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def predict_type1(img_path):
    img = image.load_img(img_path, target_size=target_size)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    preds = model.predict(x)
    pred_class = le.inverse_transform([np.argmax(preds)])[0]
    return pred_class

# 예시
# predict_type1("./PokemonData/Zubat/Zubat60.jpg")


Found 9556 validated image filenames belonging to 15 classes.
Found 2389 validated image filenames belonging to 15 classes.


/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_15637/3797898293.py:96: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(256,256,3))
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
299/299 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step - accuracy: 0.1329 - loss: 3.0479
Epoch 1: val_loss improved from None to 2.18443, saving model to best_pokemon_type1.keras
299/299 ━━━━━━━━━━━━━━━━━━━━ 115s 378ms/step - accuracy: 0.1806 - loss: 2.7132 - val_accuracy: 0.3236 - val_loss: 2.1844
Epoch 2/100
299/299 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.3001 - loss: 2.0564
Epoch 2: val_loss improved from 2.18443 to 1.85867, saving model to best_pokemon_type1.keras
299/299 ━━━━━━━━━━━━━━━━━━━━ 150s 502ms/step - accuracy: 0.3242 - loss: 1.9870 - val_accuracy: 0.4424 - val_loss: 1.8587
Epoch 3/100
299/299 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step - accuracy: 0.4011 - loss: 1.7340
Epoch 3: val_loss improved from 1.85867 to 1.70704, saving model to best_pokemon_type1.keras
299/299 ━━━━━━━━━━━━━━━━━━━━ 154s 515ms/step - accuracy: 0.4079 - loss: 1.7314 - val_accuracy: 0.4981 - val_loss: 1.7070
Epoch 4/100
299/299 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.4442 - loss: 1.6109
Epoch